# 05c - Market-Aware Betting Models

This notebook builds binary pricing models for `home_win` with a betting-oriented workflow.
The emphasis is on probability quality, calibration, and value-bet development rather than raw accuracy.

## 1. Imports and Setup

This notebook reuses the shared project helpers so that the betting-oriented workflow stays fully aligned with the core ML pipeline.
The imported modules cover four layers:
- data preparation and matchday alignment,
- model estimation and rolling backtests,
- calibration diagnostics,
- and betting-rule development.

The goal is to keep `05c` reproducible and comparable with notebooks `05b` and `08` rather than building a separate ad hoc betting script.


In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.betting_strategy import (
    select_threshold_bets,
    summarize_flat_stake_portfolio,
    sweep_binary_thresholds,
)
from src.calibration import build_calibration_summary
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.data_builder import attach_fbref_match_metadata
from src.feature_documentation import build_feature_dictionary, summarize_feature_sets
from src.feature_selection import (
    filter_features_by_correlation,
    filter_features_by_missingness,
    select_leakage_safe_features,
)
from src.ml_modeling import get_betting_model_space, get_linear_model_coefficients, run_static_feature_set_screening
from src.rolling_backtest import (
    evaluate_by_matchday,
    fit_final_ml_model,
    summarize_predictions,
    run_rolling_backtest,
)
from src.run_artifacts import save_deployment_model, save_run_artifacts

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


## 2. Load the Pricing Table

Notebook `07` must be completed first because this workflow needs both football features and market benchmark columns.
The loaded table therefore combines:
- pre-match football information from the project feature pipeline,
- and market-implied probabilities and odds from the benchmark notebook.

This is the key difference between `05c` and the earlier ML notebooks: the model is allowed to learn not only from football form variables, but also from the market view of the match.


In [3]:
RUN_KEY = 'ml_betting_binary'
TARGET_COL = 'home_win'
DEVELOPMENT_SEASON = 2024
TEST_SEASON = 2025
THRESHOLDS = [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]

pricing_path = PROCESSED_DATA_DIR / 'market_benchmark_matches.csv'
df = pd.read_csv(pricing_path)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

fbref_schedule_path = RAW_DATA_DIR / 'fbref' / 'schedule.parquet'
if not fbref_schedule_path.exists():
    fbref_schedule_path = RAW_DATA_DIR / 'fbref' / 'schedule.csv'

try:
    fbref_schedule = (
        pd.read_parquet(fbref_schedule_path)
        if fbref_schedule_path.suffix == '.parquet'
        else pd.read_csv(fbref_schedule_path)
    )
except Exception:
    fbref_schedule = pd.read_csv(RAW_DATA_DIR / 'fbref' / 'schedule.csv')

if 'matchday' not in df.columns or df['matchday'].notna().sum() == 0:
    df = df.drop(columns=[col for col in ['round', 'week', 'matchday'] if col in df.columns], errors='ignore').copy()
    df = attach_fbref_match_metadata(df, fbref_schedule)

df['away_not_lose'] = 1 - df['home_win']

print('Pricing table shape:', df.shape)
print('Development season matches:', int((df['season_id'] == DEVELOPMENT_SEASON).sum()))
print('Test season matches:', int((df['season_id'] == TEST_SEASON).sum()))
print(f"Matchday coverage: {df['matchday'].notna().mean():.1%}")
display(
    df[
        [
            'date',
            'season_id',
            'home_team',
            'away_team',
            'home_win',
            'benchmark_home_prob',
            'benchmark_away_not_lose_prob',
            'matchday',
        ]
    ].head()
)


Pricing table shape: (882, 283)
Development season matches: 306
Test season matches: 270
Matchday coverage: 85.1%


,date,season_id,home_team,away_team,home_win,benchmark_home_prob,benchmark_away_not_lose_prob,matchday
0,2023-08-18 18:30:00,2023,Werder Bremen,Bayern Munich,0,0.110254,0.889746,1
1,2023-08-19 13:30:00,2023,Bayer Leverkusen,RasenBallsport Leipzig,1,0.392757,0.607243,1
2,2023-08-19 13:30:00,2023,Wolfsburg,FC Heidenheim,1,0.585062,0.414938,1
3,2023-08-19 13:30:00,2023,Hoffenheim,Freiburg,0,0.435495,0.564505,1
4,2023-08-19 13:30:00,2023,Augsburg,Borussia M.Gladbach,0,0.343276,0.656724,1


## 3. Define Betting-Oriented Feature Sets

The feature sets are intentionally simple and interpretable.
They separate three questions:
- how far a football-only model can go without the market,
- how strong the market is on its own,
- and whether football variables still add value once market information is already available.

This structure makes the later comparison in notebook `08` much easier to interpret.


In [4]:
leakage_safe_features = select_leakage_safe_features(df)
pretest_df = df[df['season_id'] < TEST_SEASON].copy()

features_after_missing = filter_features_by_missingness(
    train_df=pretest_df,
    feature_cols=leakage_safe_features,
    max_missing_share=0.35,
)
features_after_corr, dropped_corr_features = filter_features_by_correlation(
    train_df=pretest_df,
    feature_cols=features_after_missing,
    threshold=0.95,
)

compact_football_features = [
    col for col in features_after_corr
    if (
        'elo' in col
        or 'rest_days' in col
        or 'expected_points' in col
        or 'np_xg' in col
        or 'ppda' in col
        or 'deep_completions' in col
    )
]
compact_football_features = list(dict.fromkeys(compact_football_features))

market_features = [
    col for col in [
        'benchmark_home_prob',
        'benchmark_away_not_lose_prob',
        'benchmark_home_odds',
        'benchmark_away_not_lose_fair_odds',
        'benchmark_overround',
    ]
    if col in df.columns
]

feature_sets = {
    'football_only': compact_football_features,
    'market_only': market_features,
    'football_plus_market': list(dict.fromkeys(compact_football_features + market_features)),
}

feature_set_summary = summarize_feature_sets(feature_sets)
feature_dictionary = build_feature_dictionary(
    sorted(set().union(*[set(features) for features in feature_sets.values()]))
)

display(feature_set_summary)
display(feature_dictionary.head(30))


,feature_set,n_features,families_present,family_counts_json,feature_list
0,market_only,5,Market benchmark,"{""Market benchmark"": 5}","benchmark_home_prob, benchmark_away_not_lose_p..."
1,football_only,73,"Elo strength, Expected goal difference, Expect...","{""Elo strength"": 3, ""Expected goal difference""...","home_elo_pre, away_elo_pre, elo_diff_pre, home..."
2,football_plus_market,78,"Elo strength, Expected goal difference, Expect...","{""Elo strength"": 3, ""Expected goal difference""...","home_elo_pre, away_elo_pre, elo_diff_pre, home..."


,feature,side,family,scope,horizon,description
0,away_elo_pre,Away team,Elo strength,General pre-match feature,Single-value pre-match feature,Away team; Elo strength; General pre-match fea...
1,elo_diff_pre,Home minus away,Elo strength,Relative difference,Single-value pre-match feature,Home minus away; Elo strength; Relative differ...
2,home_elo_pre,Home team,Elo strength,General pre-match feature,Single-value pre-match feature,Home team; Elo strength; General pre-match fea...
3,diff_np_xg_diff_avg_last_2_overall,Home minus away,Expected goal difference,Relative difference,Short-term form (last 2),Home minus away; Expected goal difference; Rel...
4,diff_np_xg_diff_avg_last_2_venue,Home minus away,Expected goal difference,Relative difference,Short-term form (last 2),Home minus away; Expected goal difference; Rel...
5,diff_np_xg_against_avg_last_2_overall,Home minus away,Expected goals against,Relative difference,Short-term form (last 2),Home minus away; Expected goals against; Relat...
6,diff_np_xg_against_avg_last_2_venue,Home minus away,Expected goals against,Relative difference,Short-term form (last 2),Home minus away; Expected goals against; Relat...
7,diff_np_xg_against_avg_last_8_overall,Home minus away,Expected goals against,Relative difference,Medium-term form (last 8),Home minus away; Expected goals against; Relat...
8,diff_np_xg_against_avg_last_8_venue,Home minus away,Expected goals against,Relative difference,Medium-term form (last 8),Home minus away; Expected goals against; Relat...
9,diff_np_xg_against_cum_avg_before,Home minus away,Expected goals against,Relative difference,Season-to-date form,Home minus away; Expected goals against; Relat...


## 4. Define the Betting Model Space

The model space is deliberately compact.
The purpose of `05c` is not to maximize the number of algorithms, but to produce stable probability forecasts that can be re-run quickly and compared cleanly.
For that reason the notebook keeps only fast models with usable probability outputs.


In [5]:
betting_model_space = get_betting_model_space(random_state=42)
model_run_order = list(betting_model_space.keys())
model_run_order


['logistic_regression', 'naive_bayes', 'hist_gradient_boosting']

## 5. Static Feature-Set Screening on the Development Season

Feature-set choice is fixed using the development season so that the later test-season evaluation stays cleaner.
This screening step is only a pre-selection device.
It does not provide the final performance estimate.

The primary metric here is `log_loss`, because the notebook is optimized for pricing quality rather than plain classification accuracy.


In [6]:
train_end_date = df.loc[df['season_id'] < DEVELOPMENT_SEASON, 'date'].max().strftime('%Y-%m-%d')
val_start_date = df.loc[df['season_id'] == DEVELOPMENT_SEASON, 'date'].min().strftime('%Y-%m-%d')

screening_tables = []
for model_name in model_run_order:
    screening_df = run_static_feature_set_screening(
        df=df,
        feature_sets=feature_sets,
        model_name=model_name,
        model_spec=betting_model_space[model_name],
        train_end_date=train_end_date,
        val_start_date=val_start_date,
        val_exclude_season=TEST_SEASON,
        target_col=TARGET_COL,
        primary_metric='log_loss',
    )
    screening_tables.append(screening_df)

screening_summary = pd.concat(screening_tables, ignore_index=True)
screening_summary = screening_summary.sort_values(['model', 'log_loss', 'brier_score']).reset_index(drop=True)
best_feature_map = (
    screening_summary.sort_values(['model', 'log_loss', 'brier_score'])
    .groupby('model', as_index=False)
    .first()[['model', 'feature_set']]
)
best_feature_lookup = dict(zip(best_feature_map['model'], best_feature_map['feature_set']))

display(screening_summary)
display(best_feature_map)


,model,feature_set,n_features,accuracy,macro_f1,weighted_f1,log_loss,brier_score,multiclass_brier,best_params
0,hist_gradient_boosting,market_only,5,0.643791,0.609232,0.635816,0.593584,0.205871,NaN,"{'model__learning_rate': 0.05, 'model__max_dep..."
1,hist_gradient_boosting,football_plus_market,78,0.650327,0.611265,0.639454,0.624439,0.219141,NaN,"{'model__learning_rate': 0.05, 'model__max_dep..."
2,hist_gradient_boosting,football_only,73,0.620915,0.559821,0.597335,0.637310,0.224333,NaN,"{'model__learning_rate': 0.05, 'model__max_dep..."
3,logistic_regression,market_only,5,0.683007,0.642502,0.670029,0.583486,0.200100,NaN,{'model__C': 0.3}
4,logistic_regression,football_plus_market,78,0.611111,0.583340,0.607947,0.762052,0.242711,NaN,{'model__C': 0.3}
5,logistic_regression,football_only,73,0.624183,0.598778,0.621873,0.762195,0.238688,NaN,{'model__C': 0.3}
6,naive_bayes,market_only,5,0.653595,0.625364,0.648890,0.841262,0.235820,NaN,{'model__var_smoothing': 1e-07}
7,naive_bayes,football_only,73,0.653595,0.636694,0.654619,3.474251,0.319244,NaN,{'model__var_smoothing': 1e-07}
8,naive_bayes,football_plus_market,78,0.663399,0.646449,0.664158,3.652049,0.319925,NaN,{'model__var_smoothing': 1e-07}


,model,feature_set
0,hist_gradient_boosting,market_only
1,logistic_regression,market_only
2,naive_bayes,market_only


## 6. Rolling Probability Backtests for the Development and Test Seasons

This is the main modeling step.
For each selected model and feature set, the notebook runs rolling out-of-sample predictions separately for the development season and the final test season.

The resulting prediction table stores both class predictions and probability forecasts.
That is important because the later betting layer needs probability edges relative to the market, not only binary labels.


In [7]:
season_plan = [
    ('development', DEVELOPMENT_SEASON),
    ('test', TEST_SEASON),
]

prediction_frames = []
tuning_frames = []
feature_frames = []

for model_name in model_run_order:
    selected_feature_set = best_feature_lookup[model_name]
    selected_features = feature_sets[selected_feature_set]

    for split_name, season_id in season_plan:
        predictions_df, tuning_df, feature_df = run_rolling_backtest(
            df=df,
            candidate_features=selected_features,
            model_space={model_name: betting_model_space[model_name]},
            target_col=TARGET_COL,
            test_season_id=season_id,
            validation_size=18,
            min_train_size=100,
            n_validation_windows=4,
            validation_step_size=18,
            max_missing_share=0.35,
            correlation_threshold=0.95,
            primary_metric='log_loss',
            random_state=42,
        )

        if predictions_df.empty:
            continue

        predictions_df['feature_set'] = selected_feature_set
        predictions_df['evaluation_split'] = split_name
        predictions_df['evaluation_season'] = season_id
        predictions_df['away_not_lose'] = 1 - predictions_df['home_win']
        if 'p_home_win_model' in predictions_df.columns and 'benchmark_home_prob' in predictions_df.columns:
            predictions_df['home_win_edge'] = predictions_df['p_home_win_model'] - predictions_df['benchmark_home_prob']
        if 'p_away_not_lose_model' in predictions_df.columns and 'benchmark_away_not_lose_prob' in predictions_df.columns:
            predictions_df['away_not_lose_edge'] = predictions_df['p_away_not_lose_model'] - predictions_df['benchmark_away_not_lose_prob']

        tuning_df['feature_set'] = selected_feature_set
        tuning_df['evaluation_split'] = split_name
        tuning_df['evaluation_season'] = season_id

        feature_df['feature_set'] = selected_feature_set
        feature_df['evaluation_split'] = split_name
        feature_df['evaluation_season'] = season_id
        feature_df['model'] = model_name

        prediction_frames.append(predictions_df)
        tuning_frames.append(tuning_df)
        feature_frames.append(feature_df)

all_predictions = pd.concat(prediction_frames, ignore_index=True)
all_tuning = pd.concat(tuning_frames, ignore_index=True)
all_feature_batches = pd.concat(feature_frames, ignore_index=True)

print('Predictions shape:', all_predictions.shape)
print('Tuning shape:', all_tuning.shape)
display(all_predictions.head())


Predictions shape: (1728, 38)
Tuning shape: (1011, 25)


,batch_id,date,season_id,game_id,home_win,target,target_col,y_pred,model,n_selected_features,home_team,away_team,round,week,matchday,benchmark_home_prob,benchmark_draw_prob,benchmark_away_prob,benchmark_home_not_lose_prob,benchmark_away_not_lose_prob,benchmark_home_odds,benchmark_draw_odds,benchmark_away_odds,benchmark_home_not_lose_fair_odds,benchmark_away_not_lose_fair_odds,benchmark_overround,benchmark_stage,benchmark_odds_source,prob_class_0,prob_class_1,p_home_win_model,p_away_not_lose_model,feature_set,evaluation_split,evaluation_season,away_not_lose,home_win_edge,away_not_lose_edge
0,1,2024-08-23 18:30:00,2024,27742,0,0,home_win,0,logistic_regression,4,Borussia M.Gladbach,Bayer Leverkusen,Bundesliga,1,1,0.187881,0.214421,0.597698,0.402302,0.812119,5.09,4.46,1.60,2.485694,1.231347,1.045679,closing,market_average_close,0.791104,0.208896,0.208896,0.791104,market_only,development,2024,1,0.021015,-0.021015
1,2,2024-08-24 13:30:00,2024,27743,1,1,home_win,1,logistic_regression,4,RasenBallsport Leipzig,Bochum,Bundesliga,1,1,0.784174,0.135701,0.080125,0.919875,0.215826,1.22,7.05,11.94,1.087104,4.633362,1.045268,closing,market_average_close,0.191664,0.808336,0.808336,0.191664,market_only,development,2024,0,0.024162,-0.024162
2,2,2024-08-24 13:30:00,2024,27744,1,1,home_win,1,logistic_regression,4,Hoffenheim,Holstein Kiel,Bundesliga,1,1,0.591899,0.211206,0.196895,0.803105,0.408101,1.62,4.54,4.87,1.245167,2.450375,1.042887,closing,market_average_close,0.437526,0.562474,0.562474,0.437526,market_only,development,2024,0,-0.029425,0.029425
3,2,2024-08-24 13:30:00,2024,27745,1,1,home_win,0,logistic_regression,4,Freiburg,VfB Stuttgart,Bundesliga,1,1,0.264661,0.251463,0.483876,0.516124,0.735339,3.62,3.81,1.98,1.937518,1.359918,1.043761,closing,market_average_close,0.736251,0.263749,0.263749,0.736251,market_only,development,2024,0,-0.000913,0.000913
4,2,2024-08-24 13:30:00,2024,27746,0,0,home_win,0,logistic_regression,4,Augsburg,Werder Bremen,Bundesliga,1,1,0.389521,0.270684,0.339795,0.660205,0.610479,2.46,3.54,2.82,1.514681,1.638058,1.043600,closing,market_average_close,0.631619,0.368381,0.368381,0.631619,market_only,development,2024,1,-0.021140,0.021140


## 7. Probability Results and Matchday Diagnostics

The first aggregated summary still reports the familiar classification metrics, but now it also includes probability-oriented measures such as `log_loss` and `brier_score`.
The matchday view is kept as well, because even a model with decent aggregate pricing quality may be unstable across the season.


In [8]:
result_tables = []
matchday_tables = []

for split_name, split_df in all_predictions.groupby('evaluation_split'):
    result_df = summarize_predictions(split_df, primary_metric='log_loss')
    result_df['evaluation_split'] = split_name
    result_df['feature_set'] = result_df['model'].map(best_feature_lookup)
    result_tables.append(result_df)

    matchday_df = evaluate_by_matchday(split_df)
    matchday_df['evaluation_split'] = split_name
    matchday_tables.append(matchday_df)

model_results = pd.concat(result_tables, ignore_index=True)
accuracy_by_matchday = pd.concat(matchday_tables, ignore_index=True)

development_results = model_results.loc[model_results['evaluation_split'] == 'development'].copy()
test_results = model_results.loc[model_results['evaluation_split'] == 'test'].copy()

display(model_results.sort_values(['evaluation_split', 'log_loss', 'brier_score']))
display(accuracy_by_matchday.head())


,model,accuracy,macro_f1,weighted_f1,log_loss,brier_score,multiclass_brier,evaluation_split,feature_set
0,logistic_regression,0.683007,0.649195,0.674109,0.665949,0.229138,NaN,development,market_only
1,hist_gradient_boosting,0.640523,0.587176,0.621124,0.689457,0.233492,NaN,development,market_only
2,naive_bayes,0.673203,0.647920,0.669503,1.584698,0.285004,NaN,development,market_only
3,logistic_regression,0.688889,0.658947,0.670924,0.603707,0.203453,NaN,test,market_only
4,hist_gradient_boosting,0.651852,0.635009,0.644302,0.684603,0.238284,NaN,test,market_only
5,naive_bayes,0.692593,0.673680,0.682991,1.336703,0.254041,NaN,test,market_only


,model,matchday,n_matches,accuracy,macro_f1,weighted_f1,log_loss,brier_score,multiclass_brier,evaluation_split
0,hist_gradient_boosting,1,8,0.625,0.563636,0.604545,0.623291,0.206472,NaN,development
1,hist_gradient_boosting,2,8,0.625,0.563636,0.563636,0.783664,0.246346,NaN,development
2,hist_gradient_boosting,3,8,0.750,0.666667,0.750000,0.486096,0.156211,NaN,development
3,hist_gradient_boosting,4,8,0.625,0.563636,0.604545,0.603850,0.211430,NaN,development
4,hist_gradient_boosting,5,8,0.750,0.666667,0.750000,0.603957,0.207956,NaN,development


## 8. Calibration Diagnostics

A profitable betting model does not only need informative probabilities.
It also needs probabilities that are reasonably calibrated.
This section therefore checks whether the predicted home-win and away-not-lose probabilities match the observed frequencies across probability bins.

The calibration tables are especially useful later when comparing the standard binary ML notebook with the market-aware betting notebook.


In [9]:
calibration_home_bins, calibration_home_summary = build_calibration_summary(
    predictions_df=all_predictions,
    probability_col='p_home_win_model',
    target_col='home_win',
    group_cols=['evaluation_split', 'model', 'feature_set'],
    n_bins=10,
)

calibration_away_bins, calibration_away_summary = build_calibration_summary(
    predictions_df=all_predictions,
    probability_col='p_away_not_lose_model',
    target_col='away_not_lose',
    group_cols=['evaluation_split', 'model', 'feature_set'],
    n_bins=10,
)

display(calibration_home_summary.sort_values(['evaluation_split', 'expected_calibration_error']))
display(calibration_away_summary.sort_values(['evaluation_split', 'expected_calibration_error']))


,evaluation_split,model,feature_set,n_matches,expected_calibration_error,mean_abs_bin_gap,max_abs_bin_gap,mean_predicted_prob,observed_rate
0,development,hist_gradient_boosting,market_only,306,0.064893,0.076649,0.241587,0.439451,0.385621
1,development,logistic_regression,market_only,306,0.077674,0.100732,0.185444,0.440903,0.385621
2,development,naive_bayes,market_only,306,0.110218,0.129656,0.392923,0.444483,0.385621
4,test,logistic_regression,market_only,270,0.054798,0.079820,0.250758,0.413731,0.440741
3,test,hist_gradient_boosting,market_only,270,0.072810,0.088513,0.180335,0.432204,0.440741
5,test,naive_bayes,market_only,270,0.098068,0.121356,0.254178,0.423160,0.440741


,evaluation_split,model,feature_set,n_matches,expected_calibration_error,mean_abs_bin_gap,max_abs_bin_gap,mean_predicted_prob,observed_rate
0,development,hist_gradient_boosting,market_only,306,0.064893,0.076649,0.241587,0.560549,0.614379
1,development,logistic_regression,market_only,306,0.077674,0.100732,0.185444,0.559097,0.614379
2,development,naive_bayes,market_only,306,0.110218,0.129656,0.392923,0.555517,0.614379
4,test,logistic_regression,market_only,270,0.054798,0.079820,0.250758,0.586269,0.559259
3,test,hist_gradient_boosting,market_only,270,0.072810,0.088513,0.180335,0.567796,0.559259
5,test,naive_bayes,market_only,270,0.098068,0.121356,0.254178,0.576840,0.559259


## 9. Threshold Development on the Development Season

The betting rule is developed only on the development season.
This is a methodological safeguard against choosing a threshold on the same season that is later used for the final ROI report.

The threshold grid tests how selective the strategy should be when converting model-market probability gaps into actual bets.


In [10]:
development_predictions = all_predictions.loc[all_predictions['evaluation_split'] == 'development'].copy()
strategy_rows = []

for (model_name, feature_set), group in development_predictions.groupby(['model', 'feature_set']):
    home_grid = sweep_binary_thresholds(
        predictions_df=group,
        model_probability_col='p_home_win_model',
        market_probability_col='benchmark_home_prob',
        offered_odds_col='benchmark_home_odds',
        outcome_col='home_win',
        positive_outcome=1,
        thresholds=THRESHOLDS,
        strategy_name='home_win_value',
    )
    home_grid['model'] = model_name
    home_grid['feature_set'] = feature_set
    home_grid['evaluation_split'] = 'development'
    strategy_rows.append(home_grid)

    away_grid = sweep_binary_thresholds(
        predictions_df=group,
        model_probability_col='p_away_not_lose_model',
        market_probability_col='benchmark_away_not_lose_prob',
        offered_odds_col='benchmark_away_not_lose_fair_odds',
        outcome_col='away_not_lose',
        positive_outcome=1,
        thresholds=THRESHOLDS,
        strategy_name='away_not_lose_value',
    )
    away_grid['model'] = model_name
    away_grid['feature_set'] = feature_set
    away_grid['evaluation_split'] = 'development'
    strategy_rows.append(away_grid)

strategy_development_grid = pd.concat(strategy_rows, ignore_index=True)
display(strategy_development_grid.sort_values(['strategy_name', 'roi'], ascending=[True, False]))


,strategy_name,threshold,n_bets,hit_rate,roi,total_profit,avg_edge,avg_odds,model,feature_set,evaluation_split
20,away_not_lose_value,0.04,30,0.966667,0.609957,18.298708,0.051927,1.698019,logistic_regression,market_only,development
21,away_not_lose_value,0.06,7,0.857143,0.520862,3.646033,0.067590,1.898272,logistic_regression,market_only,development
19,away_not_lose_value,0.02,110,0.754545,0.236173,25.979015,0.034832,1.734815,logistic_regression,market_only,development
35,away_not_lose_value,0.10,87,0.862069,0.218047,18.970062,0.151357,1.432132,naive_bayes,market_only,development
31,away_not_lose_value,0.02,148,0.810811,0.217304,32.160992,0.112381,1.530463,naive_bayes,market_only,development
32,away_not_lose_value,0.04,126,0.833333,0.209407,26.385327,0.126992,1.482793,naive_bayes,market_only,development
33,away_not_lose_value,0.06,116,0.844828,0.208680,24.206926,0.133759,1.455923,naive_bayes,market_only,development
34,away_not_lose_value,0.08,102,0.843137,0.203497,20.756687,0.142477,1.452622,naive_bayes,market_only,development
30,away_not_lose_value,0.00,156,0.794872,0.202590,31.604023,0.107199,1.546064,naive_bayes,market_only,development
18,away_not_lose_value,0.00,190,0.705263,0.177216,33.671131,0.025047,1.818593,logistic_regression,market_only,development


## 10. Apply the Best Development Rules to the Test Season

After the development-stage threshold search is complete, the best rules are transferred unchanged to the final test season.
This produces a cleaner out-of-sample betting result.

In other words, this is the point where the notebook stops tuning and starts genuine evaluation.


In [11]:
min_bets_required = 8
valid_development_rules = strategy_development_grid.loc[
    strategy_development_grid['n_bets'] >= min_bets_required
].copy()
if valid_development_rules.empty:
    valid_development_rules = strategy_development_grid.copy()

best_development_rules = (
    valid_development_rules
    .sort_values(['strategy_name', 'roi', 'n_bets'], ascending=[True, False, False])
    .groupby(['model', 'feature_set', 'strategy_name'], as_index=False)
    .first()
)

test_predictions = all_predictions.loc[all_predictions['evaluation_split'] == 'test'].copy()
strategy_test_rows = []

for _, rule in best_development_rules.iterrows():
    model_name = rule['model']
    feature_set = rule['feature_set']
    strategy_name = rule['strategy_name']
    threshold = float(rule['threshold'])

    candidate_df = test_predictions.loc[
        (test_predictions['model'] == model_name)
        & (test_predictions['feature_set'] == feature_set)
    ].copy()

    if strategy_name == 'home_win_value':
        bets_df = select_threshold_bets(
            predictions_df=candidate_df,
            model_probability_col='p_home_win_model',
            market_probability_col='benchmark_home_prob',
            offered_odds_col='benchmark_home_odds',
            outcome_col='home_win',
            positive_outcome=1,
            threshold=threshold,
            strategy_name=strategy_name,
        )
    else:
        bets_df = select_threshold_bets(
            predictions_df=candidate_df,
            model_probability_col='p_away_not_lose_model',
            market_probability_col='benchmark_away_not_lose_prob',
            offered_odds_col='benchmark_away_not_lose_fair_odds',
            outcome_col='away_not_lose',
            positive_outcome=1,
            threshold=threshold,
            strategy_name=strategy_name,
        )

    summary = summarize_flat_stake_portfolio(bets_df, strategy_name, threshold)
    summary['model'] = model_name
    summary['feature_set'] = feature_set
    summary['evaluation_split'] = 'test'
    strategy_test_rows.append(summary)

strategy_test_results = pd.DataFrame(strategy_test_rows)
display(best_development_rules)
display(strategy_test_results.sort_values('roi', ascending=False))


,model,feature_set,strategy_name,threshold,n_bets,hit_rate,roi,total_profit,avg_edge,avg_odds,evaluation_split
0,hist_gradient_boosting,market_only,away_not_lose_value,0.08,61,0.639344,0.141767,8.647795,0.150632,1.892799,development
1,hist_gradient_boosting,market_only,home_win_value,0.10,28,0.535714,-0.171429,-4.800000,0.126204,2.221429,development
2,logistic_regression,market_only,away_not_lose_value,0.04,30,0.966667,0.609957,18.298708,0.051927,1.698019,development
3,logistic_regression,market_only,home_win_value,0.02,48,0.604167,-0.099583,-4.780000,0.038988,1.741042,development
4,naive_bayes,market_only,away_not_lose_value,0.10,87,0.862069,0.218047,18.970062,0.151357,1.432132,development
5,naive_bayes,market_only,home_win_value,0.10,77,0.649351,-0.059740,-4.600000,0.193426,1.487922,development


,strategy_name,threshold,n_bets,hit_rate,roi,total_profit,avg_edge,avg_odds,model,feature_set,evaluation_split
3,home_win_value,0.02,50,0.700000,0.092600,4.630000,0.042895,1.564200,logistic_regression,market_only,test
1,home_win_value,0.10,53,0.603774,0.066792,3.540000,0.181671,1.847547,hist_gradient_boosting,market_only,test
4,away_not_lose_value,0.10,82,0.731707,0.043482,3.565508,0.167188,1.472183,naive_bayes,market_only,test
0,away_not_lose_value,0.08,77,0.636364,0.034986,2.693922,0.160428,1.714618,hist_gradient_boosting,market_only,test
5,home_win_value,0.10,60,0.700000,0.032667,1.960000,0.185911,1.500167,naive_bayes,market_only,test
2,away_not_lose_value,0.04,88,0.636364,0.007043,0.619809,0.081725,1.716074,logistic_regression,market_only,test


## 11. Fit the Deployment Version of the Best Development Model

The best development model is refit on the full currently available table so that the project also stores a deployment-ready artifact.
This artifact is meant for future prediction notebooks and for any live-style usage of the project.


In [12]:
best_development_row = development_results.sort_values(['log_loss', 'brier_score', 'accuracy']).iloc[0]
best_model_name = best_development_row['model']
best_feature_set = best_development_row['feature_set']
best_selected_features = feature_sets[best_feature_set]

final_fit = fit_final_ml_model(
    df=df,
    candidate_features=best_selected_features,
    model_name=best_model_name,
    model_spec=betting_model_space[best_model_name],
    target_col=TARGET_COL,
    validation_size=18,
    min_train_size=100,
    n_validation_windows=4,
    validation_step_size=18,
    max_missing_share=0.35,
    correlation_threshold=0.95,
    primary_metric='log_loss',
)

best_predictions = all_predictions.loc[
    (all_predictions['model'] == best_model_name)
    & (all_predictions['feature_set'] == best_feature_set)
    & (all_predictions['evaluation_split'] == 'test')
].copy()

print('Best development model:', best_model_name)
print('Best feature set:', best_feature_set)
display(best_predictions.head())


Best development model: logistic_regression
Best feature set: market_only


,batch_id,date,season_id,game_id,home_win,target,target_col,y_pred,model,n_selected_features,home_team,away_team,round,week,matchday,benchmark_home_prob,benchmark_draw_prob,benchmark_away_prob,benchmark_home_not_lose_prob,benchmark_away_not_lose_prob,benchmark_home_odds,benchmark_draw_odds,benchmark_away_odds,benchmark_home_not_lose_fair_odds,benchmark_away_not_lose_fair_odds,benchmark_overround,benchmark_stage,benchmark_odds_source,prob_class_0,prob_class_1,p_home_win_model,p_away_not_lose_model,feature_set,evaluation_split,evaluation_season,away_not_lose,home_win_edge,away_not_lose_edge
306,1,2025-08-22 18:30:00,2025,30224,1,1,home_win,1,logistic_regression,4,Bayern Munich,RasenBallsport Leipzig,<NA>,1,1,0.768735,0.135465,0.095800,0.904200,0.231265,1.23,6.98,9.87,1.105950,4.324050,1.057592,closing,market_average_close,0.305042,0.694958,0.694958,0.305042,market_only,test,2025,0,-0.073777,0.073777
307,2,2025-08-23 13:30:00,2025,30225,0,0,home_win,0,logistic_regression,4,FC Heidenheim,Wolfsburg,<NA>,1,1,0.344039,0.278895,0.377066,0.622934,0.655961,2.74,3.38,2.50,1.605307,1.524480,1.060821,closing,market_average_close,0.801145,0.198855,0.198855,0.801145,market_only,test,2025,1,-0.145184,0.145184
308,2,2025-08-23 13:30:00,2025,30226,0,0,home_win,0,logistic_regression,4,Bayer Leverkusen,Hoffenheim,<NA>,1,1,0.550858,0.226980,0.222162,0.777838,0.449142,1.71,4.15,4.24,1.285615,2.226467,1.061608,closing,market_average_close,0.603324,0.396676,0.396676,0.603324,market_only,test,2025,1,-0.154181,0.154181
309,2,2025-08-23 13:30:00,2025,30227,1,1,home_win,0,logistic_regression,4,Union Berlin,VfB Stuttgart,<NA>,1,1,0.241086,0.249377,0.509538,0.490462,0.758914,3.91,3.78,1.85,2.038892,1.317672,1.060845,closing,market_average_close,0.875569,0.124431,0.124431,0.875569,market_only,test,2025,0,-0.116655,0.116655
310,2,2025-08-23 13:30:00,2025,30228,1,1,home_win,0,logistic_regression,4,Eintracht Frankfurt,Werder Bremen,<NA>,1,1,0.614992,0.206800,0.178208,0.821792,0.385008,1.53,4.55,5.28,1.216853,2.597351,1.062769,closing,market_average_close,0.534069,0.465931,0.465931,0.534069,market_only,test,2025,0,-0.149061,0.149061


## 12. Interpret the Best Betting Model

This section makes the winning betting model explicit.
If the best specification is logistic regression, the notebook also extracts the coefficient table.

Because the logistic model is fitted inside a standardized pipeline, the reported coefficients should be read as the log-odds effect of a one-standard-deviation increase in the corresponding input variable.
That is why the table also reports `odds_ratio_per_1sd` to make the direction and strength easier to interpret.


In [13]:
best_model_summary = pd.DataFrame(
    {
        'item': [
            'Best model name',
            'Best feature set',
            'Primary metric',
            'Validation log loss',
            'Validation accuracy',
            'Trained through season',
            'Trained through matchday',
        ],
        'value': [
            best_model_name,
            best_feature_set,
            'log_loss',
            final_fit['fit_summary'].get('log_loss', final_fit['tuning_result'].get('log_loss')),
            final_fit['fit_summary'].get('validation_accuracy'),
            final_fit['fit_summary'].get('trained_through_season_id'),
            final_fit['fit_summary'].get('trained_through_matchday'),
        ],
    }
)

display(best_model_summary)

best_model_coefficients = pd.DataFrame()
if best_model_name == 'logistic_regression':
    best_model_coefficients = get_linear_model_coefficients(
        fitted_estimator=final_fit['estimator'],
        feature_cols=final_fit['selected_features'],
    )
    display(best_model_coefficients)
else:
    print('The winning model is not coefficient-based, so no linear coefficient table is available.')


,item,value
0,Best model name,logistic_regression
1,Best feature set,market_only
2,Primary metric,log_loss
3,Validation log loss,0.558436
4,Validation accuracy,0.708333
5,Trained through season,2025
6,Trained through matchday,30


,feature,coefficient,abs_coefficient,odds_ratio_per_1sd,direction
0,benchmark_home_prob,0.834592,0.834592,2.303874,Higher value increases home-win log-odds
1,benchmark_home_odds,-0.217195,0.217195,0.804773,Higher value decreases home-win log-odds
2,benchmark_overround,0.062409,0.062409,1.064397,Higher value increases home-win log-odds
3,benchmark_away_not_lose_fair_odds,0.014244,0.014244,1.014346,Higher value increases home-win log-odds


## 13. Save the Run Outputs

The final step persists the full run in the same artifact structure used across the project.
That includes the prediction tables, calibration summaries, strategy-development outputs, and the deployment model.

This makes notebook `05c` immediately reusable from notebook `08` without any retraining.


In [14]:
run_metadata = {
    'run_key': RUN_KEY,
    'target_col': TARGET_COL,
    'task_type': 'binary_ml_betting',
    'development_season_id': DEVELOPMENT_SEASON,
    'test_season_id': TEST_SEASON,
    'primary_metric': 'log_loss',
    'threshold_grid': THRESHOLDS,
    'model_order': model_run_order,
    'best_model_name': best_model_name,
    'best_feature_set': best_feature_set,
    'best_selected_features': best_selected_features,
    'deployment_fit_summary': final_fit['fit_summary'],
}

run_dir = save_run_artifacts(
    run_key=RUN_KEY,
    metadata=run_metadata,
    tables={
        'feature_set_summary': feature_set_summary,
        'feature_dictionary': feature_dictionary,
        'screening_summary': screening_summary,
        'all_predictions': all_predictions,
        'all_tuning': all_tuning,
        'all_feature_batches': all_feature_batches,
        'model_results': model_results,
        'accuracy_by_matchday': accuracy_by_matchday,
        'calibration_home_bins': calibration_home_bins,
        'calibration_home_summary': calibration_home_summary,
        'calibration_away_bins': calibration_away_bins,
        'calibration_away_summary': calibration_away_summary,
        'strategy_development_grid': strategy_development_grid,
        'best_development_rules': best_development_rules,
        'strategy_test_results': strategy_test_results,
        'best_model_predictions': best_predictions,
    },
)

artifact_path = save_deployment_model(
    run_key=RUN_KEY,
    artifact_name='best_model',
    model_object={
        'estimator': final_fit['estimator'],
        'selected_features': final_fit['selected_features'],
        'target_col': TARGET_COL,
        'probability_column': 'p_home_win_model',
        'model_name': best_model_name,
        'feature_set_name': best_feature_set,
    },
    metadata={
        'run_key': RUN_KEY,
        'target_col': TARGET_COL,
        'primary_metric': 'log_loss',
        'best_model_name': best_model_name,
        'best_feature_set': best_feature_set,
        'selected_features': final_fit['selected_features'],
        'fit_summary': final_fit['fit_summary'],
    },
)

print('Saved run outputs to:', run_dir)
print('Saved deployment artifact to:', artifact_path)


Saved run outputs to: C:\Users\cerve\Desktop\DP\match_prediction\data\processed\model_runs\ml_betting_binary
Saved deployment artifact to: C:\Users\cerve\Desktop\DP\match_prediction\outputs\models\deployment\ml_betting_binary\best_model.pkl
